In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import mediapipe as mp # for hand landmarker and face detector
# data visualization
import numpy as np
import matplotlib.pyplot as plt
import random
# file handling
import os
import re
from pathlib import Path
from PIL import Image
import json

In [ ]:
# set input directory, change to constant
DATASET_PATH = Path("data/")

# raise exception if missing, because i move my files around and forget i did that
if not DATASET_PATH.exists():
    raise FileNotFoundError("Dataset not found!")

# set image size constant
IMG_SIZE = 224

In [ ]:
# use keras to load dataset. we set a random seed for reporducibility and a train/val split of 80/20

# training set
train_ds = tf.keras.utils.image_dataset_from_directory(
    directory=DATASET_PATH, 
    shuffle=True, 
    verbose=True, 
    seed=12, 
    batch_size=32, 
    validation_split=0.2, 
    subset="training", 
    image_size=(IMG_SIZE, IMG_SIZE)
)

# validation set. may potentially add another set for testing
val_ds = tf.keras.utils.image_dataset_from_directory(
    directory=DATASET_PATH, 
    shuffle=True, 
    verbose=True, 
    seed=12, 
    batch_size=32, 
    validation_split=0.2, 
    subset="validation", 
    image_size=(IMG_SIZE, IMG_SIZE)
)

# show class names to confirm they loaded correctly
class_names = train_ds.class_names
print("Classes:", class_names)


### We visualize the dataset to make sure the images loaded correctly

In [ ]:
# visualize the images to make sure they loaded correctly
image_paths = []

for root, _, files in os.walk(DATASET_PATH):
    for file in files:
        if file.lower().endswith(".jpg"):
            image_paths.append(os.path.join(root, file))

# Randomly select 9 images
sample_images = random.sample(image_paths, 9)

# Display images in a 3x3 grid
fig, axes = plt.subplots(3, 3, figsize=(10, 10))

for ax, img_path in zip(axes.flatten(), sample_images):
    img = Image.open(img_path)

    # Use parent folder name as class label
    label = os.path.basename(os.path.dirname(img_path))

    ax.imshow(img)
    ax.set_title(label)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# call function to nomralize images
normalization_layer = tf.keras.layers.Rescaling(1./255)

# write functions to apply normalization on each set
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

### Similar to the above, we visualize the newly pre-processed images

In [ ]:

# visualize the preprocessed images

for images, labels in train_ds.take(1):
    plt.figure(figsize=(10, 10))
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy())
        plt.title(class_names[labels[i]])
        plt.axis("off")
    plt.show()

As input, a CNN takes tensors of shape (image_height, image_width, color_channels), ignoring the batch size. If you are new to these dimensions, color_channels refers to (R,G,B).

In [ ]:
# build the model here - input later then 3 convolutional layers 

# this is the baseline model! will add more later
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),

    # block 1
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.BatchNormalization(),

    # block 2
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2)),

    # block 3
    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2)),

    # block 4
    tf.keras.layers.Conv2D(512, 3, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2)),

    tf.keras.layers.GlobalAveragePooling2D(), # use this instead of flatten because flatten makes like 6M parameters

    # classification head
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(class_names), activation='softmax')
])

In [ ]:
# display cnn architecture
model.summary()

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

In [ ]:

history = model.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds
)

In [ ]:
print(history.history.keys())

In [ ]:
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.legend(loc='lower right')
plt.show()


In [ ]:
test_loss, test_acc = model.evaluate(val_ds, verbose=2)

In [ ]:
# quick check to make sure the cnn is actually learning from the images
# it should perform badly when the labels are shuffled
def shuffle_labels(ds):
    return ds.map(lambda x, y: (x, tf.random.shuffle(y)))

model.fit(shuffle_labels(train_ds), epochs=2)


In [ ]:
# save the model in various formats because tensorflowjs converter is finicky
model.save("model.keras")
model.save("model.h5")

In [ ]:
model.export("saved_model")

In [ ]:
# create class names to be used in the browser app

dataset_path = Path(DATASET_PATH)

class_names = sorted([
    p.name for p in dataset_path.iterdir()
    if p.is_dir()
])

data = {
    "class_names": class_names,
    "class_index": {
        name: idx for idx, name in enumerate(class_names)
    }
}

with open("tfjs_model/class_names.json", "w") as f:
    json.dump(data, f, indent=2)